
# 06 强化学习金融：核心笔记

> **本章要解决的问题**：强化学习（RL）到底是什么？它和“用历史数据训练一个预测模型”有什么不同？为什么很多 RL 交易策略回测很好看、实盘却不行？普通人和一人量化，该怎么用它、又该在什么时候不用它？

## 本章导读

这一章是 Yves Hilpisch《Reinforcement Learning for Finance: A Python-Based Introduction》的零基础笔记。不需要 TensorFlow、Gymnasium，也不需要深度学习框架——**所有代码只用 numpy 和 scipy**，把 RL 的核心思想跑到能看见结果。

按“是什么 → 怎么学 → 金融里怎么用 → 有什么坑”展开：

1. 强化学习是什么：agent、环境、状态、动作、奖励、策略、价值；
2. 从多臂老虎机到 MDP、Bellman 方程、Q-learning、DQN；
3. 金融里的四类应用：交易、对冲、资产配置、最优执行；
4. 例一：算法交易环境 + 表格 Q-learning；
5. 例二：随机、买入持有、Q 策略的评估对比；
6. 例三：动态对冲，Q-learning vs BSM delta；
7. 例四：数据增强、GBM 模拟与 KS 检验；
8. 金融 RL 的特殊问题，以及它对一人量化的含义。

**重要说明**：本章所有价格、收益、期权参数都是**合成教学数据**，为了教学效果把“可学习结构”放大了。真实市场没有这么友好，数字不能直接拿去交易。

### 学习目标

- 能用六要素（agent/环境/状态/动作/奖励/策略/价值）描述一个 RL 问题。
- 能说清 RL 和 supervised learning 的区别。
- 能解释探索与利用、MDP、Bellman 方程、Q-learning 更新公式。
- 能读懂并跑通一个表格 Q-learning 的交易小环境。
- 能对比 Q 策略、随机策略、买入持有的收益、回撤与换手。
- 能说清数据增强与模拟数据的偏差，以及金融 RL 的五个特殊问题。

### 先修

会基础统计（平均值、标准差、相关系数）；会一点 Python（numpy 基础）。不需要机器学习或金融背景。

---

## 在量化研究中的位置

- **谁用**：策略研究员、做市/对冲交易员、资产配置、执行算法研究。
- **输入**：状态特征（价格、持仓、时间、波动）、动作空间、奖励函数、模拟环境。
- **输出**：策略（状态到动作的映射）、回测/评估结果、风险与容量分析。
- **在流程的哪一步**：在“因子/信号”之后，“实盘执行”之前；它更像“策略搜索工具”，而不是预测模型。
- **失败模式**：奖励设计错、环境与真实市场不符、数据太少、过拟合、忽略成本与冲击、把回测收益当能力。
- **最小工作流**：定义环境与奖励 → 选算法 → 训练 → 样本外评估 → 压力测试 → 再谈上线。

---

### 环境准备



In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



---

## 1. 强化学习是什么

### 1.1 六个基本零件

| 概念 | 英文 | 在交易里的例子 |
|---|---|---|
| 智能体 | agent | 交易策略本身 |
| 环境 | environment | 市场、交易所、你的账户 |
| 状态 | state | 当前价格、持仓、波动、时间 |
| 动作 | action | 买入/持有/卖出（或调仓多少） |
| 奖励 | reward | 这一步赚了多少钱（可扣成本） |
| 策略 | policy | 看到状态后选动作的规则 |
| 价值 | value | 从当前状态出发，未来能拿到的累计奖励 |

一句话：**RL 是让 agent 在环境里反复试错，学会“看到什么状态、做什么动作、长期奖励最大”。**

### 1.2 和 supervised learning 的区别

| | 监督学习 | 强化学习 |
|---|---|---|
| 数据 | 有标签的样本 | 自己与环境交互产生 |
| 目标 | 预测标签 | 最大化长期奖励 |
| 反馈 | 每题都有标准答案 | 只有奖励，可能延迟、稀疏 |
| 决策 | 一次性预测 | 连续决策，动作影响后续状态 |
| 难点 | 过拟合 | 探索、奖励设计、环境偏差 |

**关键差别**：监督学习问“明天涨还是跌”，RL 问“在持仓、成本、风险的约束下，这一系列动作怎么做最划算”。所以 RL 更接近“策略搜索”，而不是“预测”。

---

## 2. 从多臂老虎机到 Q-learning

### 2.1 多臂老虎机：探索与利用

书里先用“有偏硬币/骰子”讲贝叶斯学习与概率匹配：只记历史、按出现频率下注，有偏硬币的平均收益从随机约 50 提到约 66.8；改成“按最高频下注”后约 78.8，接近理论最优 80。这说明**交互 → 更新信念 → 改进行动**就是 RL 的雏形。

想象 5 台老虎机，每台胜率不同。你每次只能拉一台，怎么在有限次数里赚最多？这就是**探索（exploration）与利用（exploitation）**的矛盾：一直拉当前最好的（利用），可能错过真正最好的；一直乱试（探索），又浪费机会。

用 Q 值记录每台机器的平均收益，三种策略：

- **随机**：完全不学；
- **贪心**：永远选当前 Q 最大的，容易“早熟”，锁死在次优臂；
- **ε-贪心**：小概率随机探索，大概率利用；
- **乐观初始化**：初始 Q 给一个很高的值，逼着策略把每台都试一遍。

### 2.2 代码：老虎机 Q-learning



In [2]:

import numpy as np
from scipy.stats import norm, ks_2samp

rng = np.random.default_rng(2202)
P = np.array([0.20, 0.35, 0.50, 0.45, 0.30])     # 5 台老虎机真实胜率（教学设定）
best = int(np.argmax(P))


def bandit(policy, n_pulls=600, seed=0):
    r = np.random.default_rng(seed)
    Q = np.zeros(len(P))
    if policy == "乐观":
        Q[:] = 1.0                                  # 乐观初始化，鼓励探索
    counts = np.zeros(len(P))
    total, opt = 0.0, 0
    for _ in range(n_pulls):
        if policy == "随机":
            a = int(r.integers(len(P)))
        elif policy == "贪心":
            a = int(np.argmax(Q))
        elif policy == "ε-贪心":
            a = int(r.integers(len(P))) if r.random() < 0.10 else int(np.argmax(Q))
        else:
            a = int(np.argmax(Q))
        reward = 1.0 if r.random() < P[a] else 0.0
        counts[a] += 1
        Q[a] += (reward - Q[a]) / counts[a]        # 样本平均
        total += reward
        opt += int(a == best)
    return total / n_pulls, opt / n_pulls


print("多臂老虎机：5 台机器真实胜率", P, "最佳", best)
print(f"{'策略':<8}{'平均收益':>9}{'选中最优臂':>10}")
for pol in ["随机", "贪心", "ε-贪心", "乐观"]:
    rs = [bandit(pol, seed=s)[0] for s in range(100)]
    os_ = [bandit(pol, seed=s)[1] for s in range(100)]
    print(f"{pol:<8}{np.mean(rs):>9.3f}{np.mean(os_):>10.0%}")


多臂老虎机：5 台机器真实胜率 [0.2  0.35 0.5  0.45 0.3 ] 最佳 2
策略           平均收益     选中最优臂


随机          0.359       20%


贪心          0.199        0%


ε-贪心        0.440       53%


乐观          0.418       41%



**读数与应用**：随机策略平均收益约 0.36；纯贪心只有 0.199、几乎从不选到最优臂——因为它一开始碰巧赢了一次就再也不换；ε-贪心 0.440，乐观初始化 0.418。**没有探索，就没有学习；但探索太多，又浪费机会。**

**常见错误**：

1. 一上来就贪心，锁死在次优动作。
2. 学习率/探索率固定不动，后期还在乱试。
3. 用一次实验的结果下结论：多跑几个随机种子再看平均。

### 2.3 从老虎机到 MDP

老虎机只有一个状态。真实决策里，**你的动作会改变状态**（买了就持仓、卖了就空仓），这就是马尔可夫决策过程（MDP）：状态 $s$、动作 $a$、转移概率 $P(s'|s,a)$、奖励 $R(s,a)$、折扣因子 $\gamma$。

目标是最大化累计折扣奖励：

$$G_t=\sum_{k=0}^{\infty}\gamma^k R_{t+k}.$$

**Bellman 方程**把“长期”拆成“现在 + 未来”：

$$V(s)=\max_a\left[R(s,a)+\gamma\sum_{s'}P(s'|s,a)V(s')\right].$$

直觉：一个状态有多好 = 现在能拿多少 + 折扣后下一状态有多好。

### 2.4 动态规划、Q-learning 与 DQN

- **动态规划（DP）**：知道转移概率时，直接迭代 Bellman 方程；
- **Q-learning**：不知道转移概率，就用“试”出来的样本更新动作价值：

$$Q(s,a)\leftarrow Q(s,a)+\alpha\left[r+\gamma\max_{a'}Q(s',a')-Q(s,a)\right].$$

其中 $\alpha$ 是学习率、$r+\gamma\max Q$ 是“目标值”、括号里是“时序差分误差”。Q-learning 是 **off-policy** 的：它学的是一条“最优路径”的价值，而实际行为可以带探索。

- **DQN**：状态太多、表格装不下时，用神经网络近似 Q。两个关键技巧：
  - **经验回放（replay buffer）**：把历史交互存起来随机抽，打破样本相关性；
  - **目标网络（target network）**：用一个更新较慢的网络算目标值，避免“自己追自己”导致发散。

本章不实现 DQN，只用表格版把思想跑通；理解了表格 Q-learning，DQN 只是“把表换成网络”。

---

## 3. 金融里的四类应用

1. **算法交易**：状态是价格/持仓/波动，动作是买/持/卖，奖励是扣费后的收益。本章例一、例二就是这个。
2. **动态对冲**：卖出期权后，状态是标的价、剩余时间、当前 delta，动作是调多少股，奖励是负的对冲误差。本章例三。
3. **动态资产配置**：在风险资产与无风险资产之间调仓。经典答案是“两基金分离”（Merton share），RL 学习的是在状态（波动、收益率）变化时怎么偏离经典解。
4. **最优执行**：大单怎么拆。Almgren–Chriss 给出“冲击 vs 风险”的解析权衡；RL（尤其 actor-critic）可以学习在盘口状态下动态调整激进程度，目标是执行成本最小。

**共同的坑**：这四类问题里，奖励都很容易设计错（只看收益、忽略风险/成本/约束），环境都很容易过于简化（无冲击、无排队、无对手方反应）。

**两个经典基线**：动态配置的“两基金分离”给出风险资产权重 $w^*=(\mu-r)/(\gamma\sigma^2)$（Merton share）；最优执行的 Almgren–Chriss 给出“冲击成本 vs 风险”的解析解。原书用 actor-critic 学连续执行力度：actor 输出动作，critic 估计价值/成本，比表格 Q 更适合连续动作。

---

## 4. 例一：算法交易环境与表格 Q-learning

### 4.1 环境设计

- **状态**：上一根收益（跌/平/涨三档）+ 当前持仓（−1/0/1）+ 趋势（价格在 20 日均线上/下）→ 18 个状态；
- **动作**：目标仓位 −1/0/+1；
- **奖励**：目标仓位 × 下一根收益 × 100 − 换仓成本；
- **学习**：Q-learning，ε 从 1.0 逐渐降到 0.05。

为简化表格法，这里用 $\gamma=0$（只看单步奖励）。真实的 RL 会用 $\gamma<1$ 处理延迟奖励。

### 4.2 代码：训练



In [3]:

COST = 0.0002
ACTIONS = np.array([-1, 0, 1])
N_STATE = 3 * 3 * 2


def make_paths(n_paths, n_steps, rng, rho=0.15, sigma=0.010, mu=0.0002):
    """教学数据：带轻微动量的收益路径（AR(1)）"""
    eps = rng.normal(0, sigma, (n_paths, n_steps))
    r = np.zeros((n_paths, n_steps))
    r[:, 0] = eps[:, 0]
    for t in range(1, n_steps):
        r[:, t] = mu + rho * (r[:, t - 1] - mu) + eps[:, t]
    price = 100 * np.cumprod(1 + r, axis=1)
    return price, r


def state_id(t, price, r, pos):
    ret = r[t]
    rb = 0 if ret < -0.004 else (1 if ret < 0.004 else 2)
    trend = 1 if price[t] > price[max(0, t - 20):t + 1].mean() else 0
    return rb * 6 + (pos + 1) * 2 + trend


def run_episode(Q, price, r, eps, alpha=0.01, gamma=0.0, train=True, rng=None):
    pos, total = 0, 0.0
    for t in range(20, len(r) - 1):
        s = state_id(t, price, r, pos)
        if train and rng.random() < eps:
            a = int(rng.integers(3))
        else:
            a = int(np.argmax(Q[s]))
        new_pos = int(ACTIONS[a])
        reward = (new_pos * r[t + 1] - COST * abs(new_pos - pos)) * 100
        s2 = state_id(t + 1, price, r, new_pos)
        if train:
            Q[s, a] += alpha * (reward + gamma * np.max(Q[s2]) - Q[s, a])
        pos, total = new_pos, total + reward
    return total


train_price, train_r = make_paths(150, 300, np.random.default_rng(2203))
Q = np.zeros((N_STATE, 3))
rng_tr = np.random.default_rng(2204)
hist = []
for epoch in range(25):
    eps = max(0.05, 1.0 * (0.85 ** epoch))
    ep_rewards = [run_episode(Q, train_price[i], train_r[i], eps, rng=rng_tr)
                  for i in range(len(train_price))]
    hist.append(np.mean(ep_rewards))

print("训练数据收益一阶自相关：", round(float(np.corrcoef(
    train_r[:, :-1].ravel(), train_r[:, 1:].ravel())[0, 1]), 3))
print("每轮平均收益（第 1/5/10/15/20/25 轮）：",
      [round(hist[i], 2) for i in [0, 4, 9, 14, 19, 24]])
print("学到的策略（动作价值 [-1, 0, 1]，只看趋势=向上）：")
for pos in [-1, 0, 1]:
    row = []
    for rb in range(3):
        s = rb * 6 + (pos + 1) * 2 + 1
        row.append(f"{['跌', '平', '涨'][rb]}:{np.round(Q[s], 2)}")
    print("  持仓", pos, "→", row)


训练数据收益一阶自相关： 0.148
每轮平均收益（第 1/5/10/15/20/25 轮）： [np.float64(-6.59), np.float64(9.79), np.float64(15.42), np.float64(17.54), np.float64(14.51), np.float64(19.23)]
学到的策略（动作价值 [-1, 0, 1]，只看趋势=向上）：
  持仓 -1 → ['跌:[-0.07 -0.02 -0.16]', '平:[-0.06 -0.02 -0.09]', '涨:[-0.17 -0.02  0.16]']
  持仓 0 → ['跌:[-0.04  0.   -0.05]', '平:[-0.17  0.   -0.04]', '涨:[-0.02  0.    0.13]']
  持仓 1 → ['跌:[-0.06 -0.02 -0.18]', '平:[-0.16 -0.02 -0.03]', '涨:[-0.26 -0.02  0.28]']



**读数与应用**：训练数据收益有 +0.148 的一阶自相关（教学设定），所以“昨天涨、今天偏涨”是可学的。训练后策略表很清楚：持仓 0、上一根“涨”时，动作 +1 的价值最高（约 0.13）；持仓 −1、上一根“涨”时应该翻多（约 0.16）；持仓 +1 时继续持有（约 0.28）。**Q-learning 从零开始学到了“顺势”策略。**

**常见错误**：

1. 状态设计太粗：只能用 3 档收益，学不到更细的规律。
2. 奖励不扣成本，策略会高频乱换仓。
3. 训练集和测试集用同一个随机种子/同一段路径。

---

## 5. 例二：策略评估——随机、买入持有、Q 策略

训练只是第一步，**能不能用要看样本外**。下面在 100 条全新的测试路径上比较三种策略：

### 5.1 代码：评估



In [4]:

test_price, test_r = make_paths(100, 260, np.random.default_rng(2205))


def positions_from(Q, price, r):
    pos, out = 0, []
    for t in range(20, len(r) - 1):
        s = state_id(t, price, r, pos)
        pos = int(ACTIONS[int(np.argmax(Q[s]))])
        out.append(pos)
    return np.array(out)


def evaluate(kind, paths_price, paths_r, Q=None):
    daily, turns, mdd = [], [], []
    for i in range(len(paths_price)):
        price, r = paths_price[i], paths_r[i]
        if kind == "Q策略":
            pos = positions_from(Q, price, r)
        elif kind == "买入持有":
            pos = np.ones(len(r) - 21)
        else:
            pos = np.random.default_rng(999 + i).choice(ACTIONS, len(r) - 21)
        ret = pos * r[21:]
        turn = np.abs(np.diff(np.r_[0, pos])).mean()
        nav = np.cumprod(1 + ret)
        daily.append(ret)
        turns.append(turn)
        mdd.append((nav / np.maximum.accumulate(nav) - 1).min())
    d = np.concatenate(daily)
    sd = d.std(ddof=1)
    return dict(ann=d.mean() * 252, vol=sd * np.sqrt(252),
                sharpe=d.mean() / sd * np.sqrt(252) if sd > 0 else 0.0,
                mdd=np.mean(mdd), turnover=np.mean(turns))


print(f"{'策略':<10}{'年化':>9}{'波动':>9}{'Sharpe':>8}{'平均最大回撤':>11}{'日均换手':>9}")
for kind in ["随机", "买入持有", "Q策略"]:
    m = evaluate(kind, test_price, test_r, Q if kind == "Q策略" else None)
    print(f"{kind:<10}{m['ann']:>9.2%}{m['vol']:>9.2%}{m['sharpe']:>8.2f}"
          f"{m['mdd']:>11.2%}{m['turnover']:>9.2f}")


策略               年化       波动  Sharpe     平均最大回撤     日均换手
随机           -2.63%   13.19%   -0.20    -14.95%     0.88
买入持有          4.63%   16.10%    0.29    -17.18%     0.00
Q策略          22.26%   11.63%    1.91     -7.38%     0.63



**读数与应用**：随机策略年化 −2.63%、Sharpe −0.20（换成成本吃掉了）；买入持有 4.63%、Sharpe 0.29；Q 策略 22.26%、Sharpe 1.91、平均最大回撤 −7.38%、日均换手 0.63。**Q 策略明显更好，但注意三点：** ①这是合成数据、动量被放大；②Q 策略的换手 0.63/天，成本再高一点收益会掉很多；③样本外只测了 100 条同分布路径，真实市场会变。

**常见错误**：

1. 只在训练集上评估，或测试集与训练集同种子。
2. 只报收益，不报回撤与换手。
3. 忘记成本：换手越高，回测和实盘差得越远。

---

## 6. 例三：动态对冲——Q-learning vs BSM delta

### 6.1 问题设置

卖出一份 1 年期看涨期权（$K=100$、$\sigma=20\%$），每周调一次对冲仓位。动作是 6 个离散的 delta 档位（0, 0.2, …, 1.0），奖励是“这一步复制误差的负数”：

$$\text{err}_t=(V_{t+1}-V_t)-\delta_t(S_{t+1}-S_t).$$

比较四种做法：**静态 0.5**、**连续 BSM delta**、**离散 BSM delta**、**Q-learning**。

### 6.2 代码：对冲学习与对比



In [5]:

K, SIGMA, TT, N_STEP = 100.0, 0.20, 1.0, 52
DELTAS = np.linspace(0, 1, 6)


def bs_call(S, Kk, tau, sigma=SIGMA):
    if tau <= 0:
        return max(S - Kk, 0.0)
    w = sigma * np.sqrt(tau)
    d1 = (np.log(S / Kk) + 0.5 * w * w) / w
    d2 = d1 - w
    return S * norm.cdf(d1) - Kk * norm.cdf(d2)


def hedge_state(S, tau):
    m = S / K
    mb = 0 if m < 0.92 else 1 if m < 0.97 else 2 if m < 1.03 else 3 if m < 1.08 else 4
    tb = min(3, int(tau / TT * 4))
    return mb * 4 + tb


def hedge_path(n_paths, seed):
    r = np.random.default_rng(seed)
    dt = TT / N_STEP
    e = r.normal(0, 1, (n_paths, N_STEP))
    inc = np.cumprod(1 + SIGMA * np.sqrt(dt) * e, axis=1)
    return np.concatenate([np.ones((n_paths, 1)), inc], axis=1) * 100


def hedge_episode(Q, S_path, eps, train, rng):
    dt = TT / N_STEP
    cum = 0.0
    for t in range(N_STEP):
        S, tau = S_path[t], TT - t * dt
        s = hedge_state(S, tau)
        if train and rng.random() < eps:
            a = int(rng.integers(len(DELTAS)))
        else:
            a = int(np.argmax(Q[s]))
        d = DELTAS[a]
        V_t, V_n = bs_call(S, K, tau), bs_call(S_path[t + 1], K, tau - dt)
        err = (V_n - V_t) - d * (S_path[t + 1] - S)
        s2 = hedge_state(S_path[t + 1], tau - dt)
        if train:
            Q[s, a] += 0.05 * (-(err * 10) ** 2 + 0.9 * np.max(Q[s2]) - Q[s, a])
        cum += err
    return cum


Qh = np.zeros((20, len(DELTAS)))
rng_h = np.random.default_rng(2206)
tr_paths = hedge_path(300, 2207)
for epoch in range(12):
    eps = max(0.05, 0.8 * (0.75 ** epoch))
    for i in range(len(tr_paths)):
        hedge_episode(Qh, tr_paths[i], eps, True, rng_h)

te_paths = hedge_path(400, 2208)


def hedge_results(kind):
    errs = []
    for S_path in te_paths:
        err = 0.0
        for t in range(N_STEP):
            S = S_path[t]
            ta = TT - t * (TT / N_STEP)
            if kind == "Q":
                d = DELTAS[int(np.argmax(Qh[hedge_state(S, ta)]))]
            elif kind == "BSM":
                w = SIGMA * np.sqrt(ta)
                d = norm.cdf((np.log(S / K) + 0.5 * w * w) / w) if ta > 0 else (1.0 if S > K else 0.0)
            elif kind == "离散BSM":
                w = SIGMA * np.sqrt(ta)
                d0 = norm.cdf((np.log(S / K) + 0.5 * w * w) / w) if ta > 0 else (1.0 if S > K else 0.0)
                d = DELTAS[int(np.argmin(np.abs(DELTAS - d0)))]
            else:
                d = 0.5
            V_t, V_n = bs_call(S, K, ta), bs_call(S_path[t + 1], K, ta - TT / N_STEP)
            err += (V_n - V_t) - d * (S_path[t + 1] - S)
        errs.append(err)
    return np.array(errs)


print("动态对冲：期末复制误差（400 条测试路径）")
print(f"{'策略':<8}{'均值绝对值':>10}{'标准差':>9}{'RMS':>9}")
for kind in ["静态0.5", "BSM", "离散BSM", "Q"]:
    e = hedge_results(kind)
    print(f"{kind:<8}{np.mean(np.abs(e)):>10.3f}{np.std(e):>9.3f}{np.sqrt(np.mean(e ** 2)):>9.3f}")


动态对冲：期末复制误差（400 条测试路径）
策略           均值绝对值      标准差      RMS


静态0.5        4.444    5.811    5.817


BSM          0.674    0.920    0.924


离散BSM        1.145    1.505    1.507


Q            1.500    1.868    1.868



**读数与应用**：静态 0.5 的 RMS 误差 5.817，几乎等于没对冲；连续 BSM delta 0.924；把 BSM delta 离散到 6 档后 1.507；Q-learning 1.868。**结论：Q 学到了“方向”，比静态好得多，但不如解析解。** 原因很直接：动作只有 6 档、状态只有 20 个、样本有限；BSM 有公式和连续 delta，本来就更强。这说明 RL 不是“越复杂越强”，**在能解析求解的问题上，解析解往往更好**。

**常见错误**：

1. 把“跑赢静态策略”当成功，其实离解析解还差很远。
2. 状态/动作太粗，学不到足够细的对冲比例。
3. 奖励设计成“越大越好”，忽略风险与方差。

---

## 7. 例四：数据增强、模拟数据与 KS 检验

金融数据太少，常用“模拟数据”来增强训练。但模拟数据如果和真实数据**分布/时序结构**不一致，策略就会学到假规律。

### 7.1 做法

- **历史数据**：带轻微动量的路径（教学设定）；
- **模拟数据**：几何布朗运动（GBM，无自相关）；
- 分别训练一个 Q 策略，在两个测试集上评估；用 **KS 检验**比较日收益分布，用**一阶自相关**比较时序结构。

原书第 5 章还用 GAN 生成时间序列，并用 KS 统计量比较生成数据与历史数据。GAN/VAE 能匹配边际分布，但不保证时序结构、尾部相关性和可交易规律；和 GBM 模拟一样，**分布像 ≠ 规律像**。

### 7.2 代码：模拟偏差实验



In [6]:

hist_tr_p, hist_tr_r = make_paths(100, 260, np.random.default_rng(2209), rho=0.15)
hist_te_p, hist_te_r = make_paths(100, 260, np.random.default_rng(2210), rho=0.15)
sim_tr_p, sim_tr_r = make_paths(100, 260, np.random.default_rng(2211), rho=0.0, sigma=0.011)
sim_te_p, sim_te_r = make_paths(100, 260, np.random.default_rng(2212), rho=0.0, sigma=0.011)


def train_q(paths_price, paths_r, seed=0, epochs=10):
    Ql = np.zeros((N_STATE, 3))
    rr = np.random.default_rng(seed)
    for epoch in range(epochs):
        eps = max(0.05, 1.0 * (0.85 ** epoch))
        for i in range(len(paths_price)):
            run_episode(Ql, paths_price[i], paths_r[i], eps, rng=rr)
    return Ql


Q_hist = train_q(hist_tr_p, hist_tr_r, seed=2213)
Q_sim = train_q(sim_tr_p, sim_tr_r, seed=2214)
ks = ks_2samp(hist_tr_r.ravel(), sim_tr_r.ravel())
ac_h = np.corrcoef(hist_tr_r[:, :-1].ravel(), hist_tr_r[:, 1:].ravel())[0, 1]
ac_s = np.corrcoef(sim_tr_r[:, :-1].ravel(), sim_tr_r[:, 1:].ravel())[0, 1]
print(f"KS 检验（日收益边际分布）：统计量 {ks.statistic:.3f}，p 值 {ks.pvalue:.1e}")
print(f"一阶自相关：历史 {ac_h:+.3f}，模拟 {ac_s:+.3f}")
print(f"{'Q策略来源':<12}{'历史测试年化':>13}{'历史Sharpe':>11}{'模拟测试年化':>13}{'模拟Sharpe':>11}")
for name, QQ in [("历史训练", Q_hist), ("模拟训练", Q_sim)]:
    m1 = evaluate("Q策略", hist_te_p, hist_te_r, QQ)
    m2 = evaluate("Q策略", sim_te_p, sim_te_r, QQ)
    print(f"{name:<12}{m1['ann']:>13.2%}{m1['sharpe']:>11.2f}{m2['ann']:>13.2%}{m2['sharpe']:>11.2f}")


KS 检验（日收益边际分布）：统计量 0.024，p 值 5.1e-07
一阶自相关：历史 +0.151，模拟 +0.007
Q策略来源              历史测试年化   历史Sharpe       模拟测试年化   模拟Sharpe
历史训练               26.00%       2.06        0.45%       0.03


模拟训练                0.00%       0.00        0.00%       0.00



**读数与应用**：KS 统计量 0.024、p 值 5.1e-7——日收益的**边际分布**差异很小，但统计上显著；真正致命的是**一阶自相关**：历史 +0.151，模拟 +0.007。结果：历史训练的 Q 策略在历史测试集上年化 26.00%、Sharpe 2.06，在模拟测试集上只有 0.45%、Sharpe 0.03；模拟训练的策略两个测试集都是空仓（0 收益）——它在无自相关的数据里找不到可交易规律，学会了“不交易”。

**一句话结论：KS 检验只看边际分布，看不出时序结构；模拟数据匹配了分布，不等于匹配了“可学习的规律”。**

**常见错误**：

1. 只用 KS 检验就宣称“模拟数据可用”。
2. 用 GBM 模拟有动量的市场，训练出的策略自然失效。
3. 把“策略在模拟数据上表现好”当“策略学会了”。

---

## 8. 金融 RL 的特殊问题与一人量化的结论

### 8.1 五个特殊问题

1. **数据少、非平稳**：市场只有一个样本，结构会变，RL 很容易过拟合；
2. **环境太理想**：无冲击、无排队、无对手方反应、可无限成交；
3. **模拟偏差**：模拟数据分布/时序与真实不同（见例四）；
4. **奖励设计**：奖励只写收益，策略就敢冒巨大风险；要写风险、成本、回撤；
5. **评估困难**：样本外只有一段，很难判断是真本事还是运气。

### 8.2 对一人量化的三句话

1. **RL 是“策略搜索工具”，不是印钞机**：它的价值在于把“状态—动作—长期收益”写清楚，帮你搜索规则；不保证找到 alpha。
2. **先做环境、奖励、评估，再谈上线**：环境（成本/冲击/约束）和奖励（风险调整）比算法选择更重要。
3. **能解析求解的，先用解析解**：BSM delta、Merton 配置、Almgren–Chriss 都是强基线；RL 只有在“有状态依赖、有摩擦、解析解失效”时才可能赢。

---

## 本章小结

### 一张表

| 概念 | 一句话 | 本章例子 |
|---|---|---|
| 六要素 | agent/环境/状态/动作/奖励/策略/价值 | 交易、对冲 |
| 探索 vs 利用 | 不试不知道，试多了浪费 | 多臂老虎机 |
| MDP/Bellman | 当前奖励 + 折扣后的未来 | 状态依赖的决策 |
| Q-learning | 用样本更新动作价值 | 表格交易策略 |
| DQN | 神经网络 + 回放 + 目标网络 | 状态太多时的扩展 |
| 金融应用 | 交易/对冲/配置/执行 | 四个小例子 |
| 模拟偏差 | 分布像 ≠ 规律像 | KS 检验 + 自相关 |
| 评估 | 样本外、成本后、风险调整 | 随机/买入持有/Q 对比 |

### 八句话

1. RL 学的是“长期奖励最大的动作序列”，不是预测标签。
2. 没有探索就没有学习，纯贪心会锁死在次优。
3. Bellman 方程把长期问题拆成“现在 + 未来”。
4. Q-learning 用样本更新 Q 值，DQN 用神经网络装下更大的状态空间。
5. 能解析求解的问题（BSM/Merton/Almgren–Chriss），解析解往往更强。
6. RL 在交易里最大的风险不是算法，而是环境与奖励设计。
7. 模拟数据匹配分布，不等于匹配时序规律。
8. RL 是策略搜索工具，不是印钞机；先做环境和评估，再谈上线。

---

## 本章习题

**1.** RL 的六要素分别是什么？在“卖出看涨期权并动态对冲”里，状态、动作、奖励各是什么？

**2.** RL 和 supervised learning 最大的区别是什么？为什么说 RL 更接近“策略搜索”？

**3.** 多臂老虎机里，纯贪心策略为什么容易失败？ε-贪心和乐观初始化分别怎么缓解？

**4.** 写出 Q-learning 的更新公式，并解释每一项的含义。

**5.** DQN 中的经验回放和目标网络各解决什么问题？

**6.** 动态对冲例子中，Q-learning 的 RMS 误差 1.868，比连续 BSM delta 的 0.924 差，但比静态 0.5 的 5.817 好。这说明什么？

**7.** 为什么 KS 检验通过不代表模拟数据可用？还应检查什么？

**8.** 金融 RL 的五个特殊问题是什么？对一人量化，最优先做哪两件事？

<details><summary>参考答案</summary>

**1.** agent、环境、状态、动作、奖励、策略/价值。对冲例中：状态=标的价与剩余时间（当前 delta），动作=调多少股，奖励=负的复制误差（或对冲成本）。

**2.** 监督学习有标签、一次性预测；RL 只有奖励、连续决策、动作改变后续状态。它优化的是“长期累计奖励”，所以是策略搜索而不是分类回归。

**3.** 贪心只看当前估计，早期偶然赢一次就锁死。ε-贪心用小概率随机尝试；乐观初始化把初始 Q 设高，迫使策略把每个动作都试一遍。

**4.** $Q(s,a)\leftarrow Q(s,a)+\alpha[r+\gamma\max_{a'}Q(s',a')-Q(s,a)]$。$\alpha$ 学习率，$r$ 即时奖励，$\gamma$ 折扣因子，$\max Q(s',a')$ 下一状态最优价值，方括号是时序差分误差。

**5.** 经验回放打破样本时间相关性、提高数据利用率；目标网络提供稳定的更新目标，避免“自己追自己”导致发散。

**6.** Q-learning 学到了方向（比静态强很多），但受离散动作、粗糙状态、有限样本限制，达不到解析解；说明 RL 不应替代能解析求解的强基线，而应作为补充或在有摩擦/状态依赖时使用。

**7.** KS 只比较边际分布；策略依赖的是时序结构与条件关系。应再检查自相关、条件分布、状态转移、以及样本外表现。

**8.** 数据少/非平稳、环境太理想、模拟偏差、奖励设计、评估困难。最优先：先把环境和奖励设计对（成本、冲击、风险），再建立严格的样本外评估。

</details>

---

## 参考资料

- Hilpisch, Y. *Reinforcement Learning for Finance: A Python-Based Introduction*. O'Reilly, 2024/2025.（原书）
- Sutton, R. S. & Barto, A. G. *Reinforcement Learning: An Introduction*. MIT Press.
- Almgren, R. & Chriss, N. (2001). *Optimal Execution of Portfolio Transactions*.
- Merton, R. C. (1969). *Lifetime Portfolio Selection under Uncertainty*.
- 本系列 `advanced/03_执行算法与微观结构`、`advanced/04_风险模型与收益归因`。
